In [1]:
import os

workdir = '/vast/ai_projects/Papenfuss_SVEnsemble2/model'
os.chdir(workdir)
os.getcwd()

'/vast/ai_projects/Papenfuss_SVEnsemble2/model'

In [2]:
import copy
import importlib

In [3]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import pyranges as pr
import sklearn

In [4]:
from sklearn.ensemble import RandomForestClassifier as RF
import sklearn.model_selection as skm
from sklearn.metrics import confusion_matrix

In [5]:
# The idea:
# 1. Split the dataset into a validation and training dataset
# 2. Train a non-traditional classifier (on the training dataset)
# 3. Estimate label frequency by applying the nontraditional classifier
# on the validation dataset and calculate class prior
# 4. Use class prior for weighing the training dataset and train again
# In Du Plessis paper there is a risk function - see nalysis of Learning from Positive and Unlabeled Data

In [6]:
df_path = 'saved/gridss_overlaps.tsv'

In [7]:
callset_overlapped_df = pd.read_csv(df_path, sep='\t', index_col='ID')

In [8]:
X = pd.get_dummies(callset_overlapped_df.drop(columns=['ref_matched']))

In [9]:
y = callset_overlapped_df['ref_matched']

In [10]:
X_train, X_val, y_train, y_val = skm.train_test_split(X, y, train_size=0.8, random_state=0)

In [11]:
RF_nontrad = RF(max_features=int(np.sqrt(X.shape[1])), random_state=0)
RF_nontrad_fit = RF_nontrad.fit(X_train, y_train)

In [12]:
y_nontrad = RF_nontrad_fit.predict(X_val)

In [13]:
sum(y_nontrad)

504

In [14]:
label_freq = sum(y_nontrad & y_val) / sum(y_val)
label_prob = sum(y) / len(y)
clpr = label_prob / label_freq
clpr

0.07154750264788205

In [15]:
# Weight for True: 2 * clpr / label_prob
# Weight for False: 1 / (1 - label_prob)

In [16]:
weight_dict = {
    0: 1 / (1 - label_prob),
    1: 2 * clpr / label_prob
}
weight_dict

{0: 1.0538781606742773, 1: 2.798994974874372}

In [17]:
RF_pu = RF(
    max_features=int(np.sqrt(X.shape[1])), random_state=0,
    class_weight=weight_dict
)
RF_pu_fit = RF_pu.fit(X_train, y_train)

In [19]:
RF_pu_fit.predict_proba(X_val)

array([[1.  , 0.  ],
       [1.  , 0.  ],
       [0.99, 0.01],
       ...,
       [1.  , 0.  ],
       [1.  , 0.  ],
       [1.  , 0.  ]])

In [40]:
y_pu = RF_pu_fit.predict(X_val)

In [41]:
sum(y_pu)

504

In [ ]:
# I am not sure it is possible to do a custom loss function in sklearn
# So I implement Elkan and Noto sample weighting scheme
# Where unlabeled samples are split into 2:
# - positive part with weight (1 - label_freq) / label_freq * y_nontrad / (1 - y_nontrad)
# - negative part with weight 1 - positive_weight

In [48]:
y_proba_nontrad = RF_nontrad.predict_proba(X_train)

In [59]:
pos_weights = y_proba_nontrad[:, 1]
pos_weights[y_train] = 1

In [60]:
# We double the unlabeled part of X and y and give negative weights to it
X_train_neg = X_train[y_train == False]
y_train_neg = y_train[y_train == False]
neg_weights = y_proba_nontrad[y_train == False, 0]

In [65]:
X_doubled = np.row_stack([X_train, X_train_neg])
y_doubled = np.concatenate([y_train, y_train_neg])
weights_doubled = np.concatenate([pos_weights, neg_weights])

In [70]:
RF_enoto = RF(
    max_features=int(np.sqrt(X.shape[1])), random_state=0
)

In [71]:
RF_enoto_fit = RF_enoto.fit(X_doubled, y_doubled, sample_weight=weights_doubled)

In [72]:
y_enoto = RF_enoto_fit.predict(X_val)

/home/users/allstaff/shikov.v/.conda/envs/SVEnsemble2/lib/python3.12/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


In [73]:
sum(y_enoto)

464